# Activity #2 — Adversarial AI: When Your Model is Compromised

**Workshop:** From Industry Security to AI Security  
**Instructor:** Cagri Temel — Hezarfen LLC  
**Format:** Team competition · 3–4 students per team · ~25 minutes

---

## The scenario

You are the AI team at an airline. The security team (Sundar's team in the morning session) flags an incident — an attacker has compromised one of engine 17's sensors:

> *"A sensor on **engine 17** has been spoofed. We see drift, stuck-at, or noise-injection patterns across three different test files — but we don't know **which** sensor. Determine the affected sensor before the next flight."*

This is a classic **adversarial AI** problem: the model still produces predictions, but the input has been corrupted in a way the model wasn't trained to handle. Your defense is **explainability** — using the model's own decision logic to localize the attack.

You have three suspect data files: `attack_A.csv`, `attack_B.csv`, `attack_C.csv`.

Each is a copy of clean engine 17 telemetry — but **one sensor channel** has been manipulated.

## Your tools

1. The `SoftDecisionTree` from Activity #1 (re-trained inline here so you don't depend on the previous notebook)
2. A `RandomForestClassifier` baseline

## Goal

For each of the three attack files, identify:
- **Which sensor** (s1, s2, …, s21) was manipulated?
- **What type** of attack (drift / stuck-at / noise)?

First team to get all three correct wins. 🏆


## Step 0 — Setup (run all cells through Step 3 without changes)

In [ ]:
!pip install -q neural-trees pandas matplotlib seaborn scikit-learn

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from neural_trees import SoftDecisionTree

In [ ]:
BASE = 'https://raw.githubusercontent.com/cgrtml/wsu-workshop-may15/main/data'
for f in ['train_FD001.txt', 'engine17_clean.csv', 'attack_A.csv', 'attack_B.csv', 'attack_C.csv']:
    !wget -q -nc {BASE}/{f}
print('Files downloaded.')

## Step 1 — Re-train the models on clean data

Same setup as Activity #1, plus a RandomForest baseline.

In [ ]:
cols = ['engine_id', 'cycle'] + [f'op{i}' for i in range(1, 4)] + [f's{i}' for i in range(1, 22)]
train = pd.read_csv('train_FD001.txt', sep=r'\s+', header=None, names=cols)
max_cycles = train.groupby('engine_id')['cycle'].max().rename('max_cycle')
train = train.merge(max_cycles, on='engine_id')
train['RUL'] = (train['max_cycle'] - train['cycle']).clip(upper=125)

def bin_rul(r):
    if r < 30: return 0
    if r < 80: return 1
    return 2

train['health'] = train['RUL'].apply(bin_rul)

all_sensors = [f's{i}' for i in range(1, 22)]
informative = [s for s in all_sensors if train[s].std() > 0.001]
scaler = StandardScaler().fit(train[informative].values)
X = scaler.transform(train[informative].values)
y = train['health'].values

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

tree = SoftDecisionTree(depth=4, max_epochs=30, learning_rate=0.01, penalty_coef=1e-3, verbose=False)
tree.fit(X_tr, y_tr)
rf = RandomForestClassifier(n_estimators=100, random_state=42).fit(X_tr, y_tr)

print(f'SoftDecisionTree test accuracy: {(tree.predict(X_te) == y_te).mean():.3f}')
print(f'RandomForest    test accuracy: {(rf.predict(X_te) == y_te).mean():.3f}')

labels = ['Critical', 'Caution', 'Healthy']

## Step 2 — Helper to score a file

Same scaler as training. We compare each attack file's predictions to the clean baseline.

In [ ]:
def predict_engine(file_path, model):
    df = pd.read_csv(file_path)
    Xq = scaler.transform(df[informative].values)
    return df, model.predict(Xq), model.predict_proba(Xq)

# Sanity check on clean engine 17 — both models should agree mostly
_, clean_tree_pred, _ = predict_engine('engine17_clean.csv', tree)
_, clean_rf_pred, _ = predict_engine('engine17_clean.csv', rf)
print('Clean engine 17 — soft tree vs random forest agreement:',
      f'{(clean_tree_pred == clean_rf_pred).mean():.2f}')

## Step 3 — Compare attack predictions to clean baseline

In [ ]:
def attack_summary(attack_file):
    _, p_tree, _ = predict_engine(attack_file, tree)
    _, p_rf, _ = predict_engine(attack_file, rf)
    tree_diff = (p_tree != clean_tree_pred).sum()
    rf_diff = (p_rf != clean_rf_pred).sum()
    print(f'{attack_file}:')
    print(f'   Soft Tree predictions changed at {tree_diff:3d} / {len(p_tree)} cycles')
    print(f'   RandomForest predictions changed at {rf_diff:3d} / {len(p_rf)} cycles')

for f in ['attack_A.csv', 'attack_B.csv', 'attack_C.csv']:
    attack_summary(f)
    print()

## Step 4 — INVESTIGATE

Now it's your turn. The cells below give you tools — use them to figure out which sensor was manipulated in each attack.

### Tool A — visualize each sensor in the attack file vs the clean file

In [ ]:
def diff_plot(attack_file, sensors=None):
    clean = pd.read_csv('engine17_clean.csv')
    attack = pd.read_csv(attack_file)
    if sensors is None:
        sensors = informative
    n = len(sensors)
    cols_per_row = 4
    rows = (n + cols_per_row - 1) // cols_per_row
    fig, axes = plt.subplots(rows, cols_per_row, figsize=(16, 2.5 * rows))
    for ax, s in zip(axes.flat, sensors):
        ax.plot(clean['cycle'], clean[s], label='clean', alpha=0.7)
        ax.plot(attack['cycle'], attack[s], label='attack', alpha=0.7)
        ax.set_title(s)
    for ax in axes.flat[n:]:
        ax.axis('off')
    axes.flat[0].legend()
    plt.tight_layout()
    plt.suptitle(attack_file, y=1.02)
    plt.show()

# Look at all sensors at once for attack A
diff_plot('attack_A.csv')

### TODO — investigate Attack B and Attack C

Use `diff_plot()` on the other two files. Which sensor's blue and orange lines diverge?


In [ ]:
# TODO: diff_plot('attack_B.csv')



In [ ]:
# TODO: diff_plot('attack_C.csv')



### Tool B — quantify per-sensor divergence

If your eye missed it, this ranks sensors by how much they differ between clean and attack.

In [ ]:
def rank_divergence(attack_file):
    clean = pd.read_csv('engine17_clean.csv')
    attack = pd.read_csv(attack_file)
    diffs = {}
    for s in informative:
        diffs[s] = float(np.mean(np.abs(clean[s].values - attack[s].values)))
    return sorted(diffs.items(), key=lambda kv: -kv[1])

for f in ['attack_A.csv', 'attack_B.csv', 'attack_C.csv']:
    print(f'--- {f} ---')
    for sensor, score in rank_divergence(f)[:3]:
        print(f'   {sensor:4s}  mean |diff| = {score:.4f}')
    print()

### Tool C — explanation comparison

How do the two models behave under attack? The point of the workshop:

- The **RandomForest** changes its prediction. But it cannot tell you which sensor caused the change.
- The **SoftDecisionTree's split weights** show *which sensor was driving the decision* — and that fingerprint shifts when the attack hits.

Look at the dominant sensor at the root and child nodes:

In [ ]:
splits = tree.get_split_weights()
print('Dominant sensor at each internal node of your soft tree:')
for i, w in enumerate(splits):
    feat = np.argmax(np.abs(w))
    print(f'  Node {i:2d} → {informative[feat]}')

## Step 5 — Submit your team's answers

Fill in the dictionary below and run the cell. The instructor will collect.


In [ ]:
TEAM_NAME = ''  # e.g., 'Team Cougar'

answers = {
    'attack_A': {'sensor': 's??', 'type': 'drift / stuck-at / noise'},
    'attack_B': {'sensor': 's??', 'type': 'drift / stuck-at / noise'},
    'attack_C': {'sensor': 's??', 'type': 'drift / stuck-at / noise'},
}

print(f'Team: {TEAM_NAME}')
for k, v in answers.items():
    print(f'  {k}: sensor={v["sensor"]:4s}  type={v["type"]}')

## Wrap-up question

Discuss with your team:

> *In a real production system — say, a payments fraud detector or an aircraft predictive-maintenance model — would you rely on the model's prediction, on its explanation, or both? When would you trust one over the other? How does this connect to the security threats Sundar described this morning?*

Be ready to answer in the wrap-up.
